In [1]:
#UNCHANGED
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from scipy.stats import randint, uniform
import category_encoders as ce

In [2]:
#UNCHANGED
data_path = Path("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data")

# Load pre-encoding dataset
df = pd.read_csv(data_path / "dataset_pre_encoding.csv")
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (564706, 26)


In [3]:
#UNCHANGED
# Chronological 70/15/15 split by year
TARGET    = "AttendanceTimeSeconds"
DROP_COLS = ["index", "IncidentNumber", "avg_speed", "PumpOrder"]

train_df = df[df["CalYear"] <= 2023].copy()
test_df  = df[df["CalYear"] == 2024].copy()
val_df   = df[df["CalYear"] >= 2025].copy()

n = len(df)
print(f"Train: {len(train_df)} ({len(train_df)/n*100:.1f}%) | Years: 2021-2023")
print(f"Test:  {len(test_df)} ({len(test_df)/n*100:.1f}%) | Years: 2024")
print(f"Val:   {len(val_df)} ({len(val_df)/n*100:.1f}%) | Years: 2025-2026")

def get_data(data):
    to_drop = [TARGET] + DROP_COLS
    X = data.drop(columns=[c for c in to_drop if c in data.columns])
    y = data[TARGET]
    y_log = np.log1p(y)
    return X, y, y_log

X_train, y_train, y_train_log = get_data(train_df)
X_test,  y_test,  y_test_log  = get_data(test_df)
X_val,   y_val,   y_val_log   = get_data(val_df)

print(f"\nFeatures: {list(X_train.columns)}")

Train: 310660 (55.0%) | Years: 2021-2023
Test:  116221 (20.6%) | Years: 2024
Val:   137825 (24.4%) | Years: 2025-2026

Features: ['CalYear', 'Month', 'Weekday', 'Hour', 'Is_Nightshift', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Public_Holiday', 'DeployedFromStation_Name', 'IncidentGroup', 'Is_SpecialService', 'SpecialServiceType', 'PropertyCategory', 'PropertyType', 'IncGeo_BoroughName', 'Is_RepeatedCall', 'Latitude', 'Longitude', 'distance_fire_to_station', 'NumOfCalls_bucket', 'Is_central_London']


In [4]:
#CHANGED
# CHANGE: apply_encoding converted into sklearn Transformer
# This allows encoding to be recalculated inside each CV fold
# preventing data leakage from LOE encoding

feature_config = {
    "Month":             {"encoding": "CYCLIC"},
    "Weekday":           {"encoding": "CYCLIC"},
    "Hour":              {"encoding": "CYCLIC"},
    "CalYear":           {"encoding": "NUMERIC_KEEP"},
    "Is_Nightshift":     {"encoding": "BINARY_KEEP"},
    "Is_Rush_Hour":      {"encoding": "BINARY_KEEP"},
    "Is_Weekend":        {"encoding": "BINARY_KEEP"},
    "Is_Public_Holiday": {"encoding": "BINARY_KEEP"},
    "Is_SpecialService": {"encoding": "BINARY_KEEP"},
    "Is_RepeatedCall":   {"encoding": "BINARY_KEEP"},
    "Is_central_London": {"encoding": "BINARY_KEEP"},
    "Latitude":          {"encoding": "NUMERIC_KEEP"},
    "Longitude":         {"encoding": "NUMERIC_KEEP"},
    "distance_fire_to_station": {"encoding": "NUMERIC_KEEP"},
    "IncidentGroup":     {"encoding": "ONE_HOT"},
    "PropertyCategory":  {"encoding": "ONE_HOT"},
    "NumOfCalls_bucket": {"encoding": "ONE_HOT"},
    "SpecialServiceType":{"encoding": "TOP_N_PLUS_ONE_HOT", "top_n": 10},
    "DeployedFromStation_Name": {"encoding": "LEAVE_ONE_OUT_TARGET"},  # LOE — needs Pipeline
    "PropertyType":      {"encoding": "LEAVE_ONE_OUT_TARGET"},         # LOE — needs Pipeline
    "IncGeo_BoroughName":{"encoding": "LEAVE_ONE_OUT_TARGET"}          # LOE — needs Pipeline
}

class EncodingTransformer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer that wraps apply_encoding.
    By being inside a Pipeline, it re-fits on each CV fold's
    training data — preventing LOE target encoding data leakage.
    """
    def __init__(self, config=feature_config):
        self.config = config
        self.fitted_encoders_ = {}

    def fit(self, X, y=None):
        # CHANGED: fit is called fresh on each CV fold's training data
        self.fitted_encoders_ = {}
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X

        for col, cfg in self.config.items():
            if col not in X.columns:
                continue
            method = cfg["encoding"]

            if method in ["ONE_HOT", "TOP_N_PLUS_ONE_HOT"]:
                data_to_fit = X[[col]].copy()
                top_n_list = None
                if method == "TOP_N_PLUS_ONE_HOT":
                    top_n_list = X[col].value_counts().head(cfg.get("top_n", 10)).index.tolist()
                    data_to_fit[col] = data_to_fit[col].where(data_to_fit[col].isin(top_n_list), "Other")
                enc = ce.OneHotEncoder(cols=[col], use_cat_names=True).fit(data_to_fit)
                self.fitted_encoders_[col] = (enc, top_n_list)

            elif method == "LEAVE_ONE_OUT_TARGET":
                # CHANGED: LOE fitted on fold's y — no leakage from validation fold
                enc = ce.LeaveOneOutEncoder(cols=[col]).fit(X[[col]], y)
                self.fitted_encoders_[col] = enc

        return self

    def transform(self, X, y=None):
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        encoded_parts = []

        for col, cfg in self.config.items():
            if col not in X.columns:
                continue
            method = cfg["encoding"]

            if method in ["NUMERIC_KEEP", "BINARY_KEEP"]:
                encoded_parts.append(X[[col]])

            elif method == "CYCLIC":
                periods = {"Month": 12, "Weekday": 7, "Hour": 24}
                p = periods[col]
                encoded_parts.append(pd.DataFrame({
                    f"{col}_sin": np.sin(2 * np.pi * X[col] / p),
                    f"{col}_cos": np.cos(2 * np.pi * X[col] / p)
                }, index=X.index))

            elif method in ["ONE_HOT", "TOP_N_PLUS_ONE_HOT"]:
                enc, top_n_list = self.fitted_encoders_[col]
                data_to_transform = X[[col]].copy()
                if top_n_list:
                    data_to_transform[col] = data_to_transform[col].where(
                        data_to_transform[col].isin(top_n_list), "Other"
                    )
                encoded_parts.append(enc.transform(data_to_transform))

            elif method == "LEAVE_ONE_OUT_TARGET":
                encoded_parts.append(self.fitted_encoders_[col].transform(X[[col]]))

        return pd.concat(encoded_parts, axis=1).reset_index(drop=True)

In [5]:
#CHANGED
# CHANGE: Pipeline wraps encoder + scaler + model
# TimeSeriesSplit used in all CV — respects temporal order
tscv = TimeSeriesSplit(n_splits=5)

def make_pipeline(model):
    return Pipeline([
        ('encoder', EncodingTransformer()),   # CHANGED: encoding inside pipeline
        ('scaler',  StandardScaler()),
        ('model',   model)
    ])

In [6]:
#CHANGED
# CHANGE: models now use Pipeline — fit on raw X (not pre-encoded)
# Baseline models
modelos_basicos = {
    "Linear Regression": LinearRegression(),
    "KNN":               KNeighborsRegressor(n_jobs=-1),
    "Random Forest":     RandomForestRegressor(random_state=42, n_jobs=-1)
}

results_baseline = []

for name, model in modelos_basicos.items():
    print(f"Training {name}...")
    pipe = make_pipeline(model)    # CHANGED: wrapped in pipeline
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results_baseline.append({
        'Model': name,
        'R²':    r2_score(y_test, y_pred),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred))
    })
    print(f"  R²: {results_baseline[-1]['R²']:.4f} | MAE: {results_baseline[-1]['MAE']:.4f} | RMSE: {results_baseline[-1]['RMSE']:.4f}")

Training Linear Regression...
  R²: 0.5069 | MAE: 56.3054 | RMSE: 76.7839
Training KNN...
  R²: 0.3098 | MAE: 69.3095 | RMSE: 90.8418
Training Random Forest...
  R²: 0.5526 | MAE: 52.4020 | RMSE: 73.1345


In [7]:
#CHANGED
# CHANGE: advanced models use Pipeline
modelos_avanzados = {
    "XGBoost":  XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'),
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(random_seed=42, verbose=0,  train_dir='/tmp/catboost')
}

results_avanzados = []

for name, model in modelos_avanzados.items():
    print(f"Training {name}...")
    pipe = make_pipeline(model)    # CHANGED: wrapped in pipeline
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results_avanzados.append({
        'Model': name,
        'R²':    r2_score(y_test, y_pred),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred))
    })
    print(f"  R²: {results_avanzados[-1]['R²']:.4f} | MAE: {results_avanzados[-1]['MAE']:.4f} | RMSE: {results_avanzados[-1]['RMSE']:.4f}")

Training XGBoost...
  R²: 0.5695 | MAE: 50.8892 | RMSE: 71.7413
Training LightGBM...


/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  R²: 0.5600 | MAE: 51.7534 | RMSE: 72.5328
Training CatBoost...
  R²: 0.5739 | MAE: 50.5769 | RMSE: 71.3711


In [8]:
#CHANGED
# CHANGE: GridSearch params prefixed with 'model__' for Pipeline
# CHANGE: cv=tscv — TimeSeriesSplit instead of random cv
# GridSearch — basic models
param_grids = {
    "Linear Regression": {
        'model':   LinearRegression(),
        'params':  {},
        'use_log': True
    },
    "KNN": {
        'model':   KNeighborsRegressor(n_jobs=-1),
        'params':  {
            'model__n_neighbors': [15, 19, 25],   # CHANGED: model__ prefix
            'model__weights':     ['distance']
        },
        'use_log': True
    },
    "Random Forest": {
        'model':   RandomForestRegressor(random_state=42, n_jobs=-1, max_samples=0.3),
        'params':  {
            'model__n_estimators':      [100],     # CHANGED: model__ prefix
            'model__max_depth':         [10, 20],
            'model__min_samples_split': [5]
        },
        'use_log': False
    }
}

results_grid_basic = []

for name, cfg in param_grids.items():
    print(f"\nTraining {name} with GridSearch...")
    y_tr = y_train_log if cfg['use_log'] else y_train
    pipe = make_pipeline(cfg['model'])
    grid = GridSearchCV(
        pipe, cfg['params'],
        cv=tscv,              # CHANGED: TimeSeriesSplit
        scoring='r2', n_jobs=-1
    )
    grid.fit(X_train, y_tr)
    y_pred = grid.predict(X_test)
    if cfg['use_log']:
        y_pred = np.exp(y_pred)
    results_grid_basic.append({
        'Model': f"{name} (GridSearch)",
        'R²':    r2_score(y_test, y_pred),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred))
    })
    print(f"  Best params: {grid.best_params_ if grid.best_params_ else '---'}")
    print(f"  CV R² mean:  {grid.best_score_:.4f}")
    print(f"  Test R²: {results_grid_basic[-1]['R²']:.4f} | MAE: {results_grid_basic[-1]['MAE']:.4f}")


Training Linear Regression with GridSearch...
  Best params: ---
  CV R² mean:  0.4978
  Test R²: 0.3950 | MAE: 58.7607

Training KNN with GridSearch...
  Best params: {'model__n_neighbors': 19, 'model__weights': 'distance'}
  CV R² mean:  0.3490
  Test R²: 0.3455 | MAE: 66.3676

Training Random Forest with GridSearch...
  Best params: {'model__max_depth': 20, 'model__min_samples_split': 5, 'model__n_estimators': 100}
  CV R² mean:  0.5567
  Test R²: 0.5553 | MAE: 52.1766


In [9]:
#CHANGED
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# GridSearch — XGBoost
print("Training XGBoost with GridSearch...")
pipe_xgb = make_pipeline(XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'))
grid_xgb = GridSearchCV(
    pipe_xgb,
    {
        'model__n_estimators':  [100, 200],    # CHANGED: model__ prefix
        'model__max_depth':     [6, 10],
        'model__learning_rate': [0.05, 0.1]
    },
    cv=tscv, scoring='r2', n_jobs=-1           # CHANGED: TimeSeriesSplit
)
grid_xgb.fit(X_train, y_train)
y_pred_xgb = grid_xgb.predict(X_test)
result_xgb_grid = {
    'Model': 'XGBoost (GridSearch)',
    'R²':    r2_score(y_test, y_pred_xgb),
    'MAE':   mean_absolute_error(y_test, y_pred_xgb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_xgb))
}
print(f"  Best params: {grid_xgb.best_params_}")
print(f"  CV R² mean:  {grid_xgb.best_score_:.4f}")
print(f"  Test R²: {result_xgb_grid['R²']:.4f} | MAE: {result_xgb_grid['MAE']:.4f}")

Training XGBoost with GridSearch...


/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Best params: {'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 200}
  CV R² mean:  0.5731
  Test R²: 0.5681 | MAE: 51.0381


In [10]:
#CHANGED
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# GridSearch — LightGBM
print("Training LightGBM with GridSearch...")
pipe_lgbm = make_pipeline(
    LGBMRegressor(random_state=42, n_jobs=2, verbose=-1,
                  num_leaves=31, subsample=0.8, colsample_bytree=0.8)
)
grid_lgbm = GridSearchCV(
    pipe_lgbm,
    {
        'model__n_estimators':  [100],         # CHANGED: model__ prefix
        'model__max_depth':     [6, 10],
        'model__learning_rate': [0.1]
    },
    cv=tscv, scoring='r2', n_jobs=4           # CHANGED: TimeSeriesSplit
)
grid_lgbm.fit(X_train, y_train)
y_pred_lgbm = grid_lgbm.predict(X_test)
result_lgbm_grid = {
    'Model': 'LightGBM (GridSearch)',
    'R²':    r2_score(y_test, y_pred_lgbm),
    'MAE':   mean_absolute_error(y_test, y_pred_lgbm),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
}
print(f"  Best params: {grid_lgbm.best_params_}")
print(f"  CV R² mean:  {grid_lgbm.best_score_:.4f}")
print(f"  Test R²: {result_lgbm_grid['R²']:.4f} | MAE: {result_lgbm_grid['MAE']:.4f}")

Training LightGBM with GridSearch...


/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/valid

  Best params: {'model__learning_rate': 0.1, 'model__max_depth': 10, 'model__n_estimators': 100}
  CV R² mean:  0.5677
  Test R²: 0.5588 | MAE: 51.8351


In [11]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# GridSearch — CatBoost
print("Training CatBoost with GridSearch...")
pipe_cb = make_pipeline(CatBoostRegressor(random_seed=42, verbose=0,  train_dir='/tmp/catboost'))
grid_cb = GridSearchCV(
    pipe_cb,
    {
        'model__iterations':    [100, 200],    # CHANGED: model__ prefix
        'model__depth':         [6, 10],
        'model__learning_rate': [0.05, 0.1]
    },
    cv=tscv, scoring='r2', n_jobs=-1           # CHANGED: TimeSeriesSplit
)
grid_cb.fit(X_train, y_train)
y_pred_cb = grid_cb.predict(X_test)
result_cb_grid = {
    'Model': 'CatBoost (GridSearch)',
    'R²':    r2_score(y_test, y_pred_cb),
    'MAE':   mean_absolute_error(y_test, y_pred_cb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_cb))
}
print(f"  Best params: {grid_cb.best_params_}")
print(f"  CV R² mean:  {grid_cb.best_score_:.4f}")
print(f"  Test R²: {result_cb_grid['R²']:.4f} | MAE: {result_cb_grid['MAE']:.4f}")

Training CatBoost with GridSearch...
  Best params: {'model__depth': 10, 'model__iterations': 200, 'model__learning_rate': 0.1}
  CV R² mean:  0.5722
  Test R²: 0.5674 | MAE: 51.1424


In [12]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# RandomizedSearch — XGBoost
print("Training XGBoost with RandomizedSearchCV...")
random_xgb = RandomizedSearchCV(
    make_pipeline(XGBRegressor(random_state=42, n_jobs=2, tree_method='hist')),
    {
        'model__n_estimators':     randint(100, 400),   # CHANGED: model__ prefix
        'model__max_depth':        randint(4, 12),
        'model__learning_rate':    uniform(0.01, 0.2),
        'model__subsample':        uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4)
    },
    n_iter=20, cv=tscv,                                 # CHANGED: TimeSeriesSplit
    scoring='r2', n_jobs=4, random_state=42
)
random_xgb.fit(X_train, y_train)
y_pred_rxgb = random_xgb.predict(X_test)
result_xgb_random = {
    'Model': 'XGBoost (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rxgb),
    'MAE':   mean_absolute_error(y_test, y_pred_rxgb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rxgb))
}
print(f"  Best params: {random_xgb.best_params_}")
print(f"  CV R² mean:  {random_xgb.best_score_:.4f}")
print(f"  Test R²: {result_xgb_random['R²']:.4f} | MAE: {result_xgb_random['MAE']:.4f}")

Training XGBoost with RandomizedSearchCV...
  Best params: {'model__colsample_bytree': np.float64(0.7554709158757928), 'model__learning_rate': np.float64(0.06426980635477918), 'model__max_depth': 8, 'model__n_estimators': 379, 'model__subsample': np.float64(0.7427013306774357)}
  CV R² mean:  0.5760
  Test R²: 0.5776 | MAE: 50.2285


In [13]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# RandomizedSearch — LightGBM
print("Training LightGBM with RandomizedSearchCV...")
random_lgbm = RandomizedSearchCV(
    make_pipeline(LGBMRegressor(random_state=42, n_jobs=2, verbose=-1)),
    {
        'model__n_estimators':     randint(100, 400),   # CHANGED: model__ prefix
        'model__max_depth':        randint(4, 12),
        'model__learning_rate':    uniform(0.01, 0.2),
        'model__num_leaves':       randint(20, 100),
        'model__subsample':        uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4)
    },
    n_iter=20, cv=tscv,                                 # CHANGED: TimeSeriesSplit
    scoring='r2', n_jobs=4, random_state=42
)
random_lgbm.fit(X_train, y_train)
y_pred_rlgbm = random_lgbm.predict(X_test)
result_lgbm_random = {
    'Model': 'LightGBM (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rlgbm),
    'MAE':   mean_absolute_error(y_test, y_pred_rlgbm),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rlgbm))
}
print(f"  Best params: {random_lgbm.best_params_}")
print(f"  CV R² mean:  {random_lgbm.best_score_:.4f}")
print(f"  Test R²: {result_lgbm_random['R²']:.4f} | MAE: {result_lgbm_random['MAE']:.4f}")

Training LightGBM with RandomizedSearchCV...


/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/utils/valid

  Best params: {'model__colsample_bytree': np.float64(0.6733618039413735), 'model__learning_rate': np.float64(0.07084844859190755), 'model__max_depth': 9, 'model__n_estimators': 352, 'model__num_leaves': 68, 'model__subsample': np.float64(0.8099098641033556)}
  CV R² mean:  0.5778
  Test R²: 0.5731 | MAE: 50.6355


In [14]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: model__ prefix on params, cv=tscv, Pipeline
# RandomizedSearch — CatBoost
print("Training CatBoost with RandomizedSearchCV...")
random_cb = RandomizedSearchCV(
    make_pipeline(CatBoostRegressor(random_seed=42, verbose=0,  train_dir='/tmp/catboost')),
    {
        'model__iterations':    randint(100, 400),      # CHANGED: model__ prefix
        'model__depth':         randint(4, 10),
        'model__learning_rate': uniform(0.01, 0.2),
        'model__l2_leaf_reg':   randint(1, 10)
    },
    n_iter=20, cv=tscv,                                 # CHANGED: TimeSeriesSplit
    scoring='r2', n_jobs=-1, random_state=42
)
random_cb.fit(X_train, y_train)
y_pred_rcb = random_cb.predict(X_test)
result_cb_random = {
    'Model': 'CatBoost (RandomizedSearch)',
    'R²':    r2_score(y_test, y_pred_rcb),
    'MAE':   mean_absolute_error(y_test, y_pred_rcb),
    'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred_rcb))
}
print(f"  Best params: {random_cb.best_params_}")
print(f"  CV R² mean:  {random_cb.best_score_:.4f}")
print(f"  Test R²: {result_cb_random['R²']:.4f} | MAE: {result_cb_random['MAE']:.4f}")

Training CatBoost with RandomizedSearchCV...
  Best params: {'model__depth': 7, 'model__iterations': 370, 'model__l2_leaf_reg': 8, 'model__learning_rate': np.float64(0.12973169683940733)}
  CV R² mean:  0.5754
  Test R²: 0.5698 | MAE: 50.9355


In [15]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: validation set evaluation on best model
# Full model comparison
df_final = pd.concat([
    pd.DataFrame([
        result_xgb_random, result_lgbm_random, result_cb_random,
        result_xgb_grid,   result_lgbm_grid,   result_cb_grid
    ]),
    pd.DataFrame(results_avanzados),
    pd.DataFrame(results_baseline)
]).sort_values('R²', ascending=False).round(4).reset_index(drop=True)

print("\nFull model comparison (test set):")
print(df_final.to_string(index=False))

# CHANGED: final evaluation on validation set — only once on best model
best_model = random_xgb   # update after seeing results
y_pred_val = best_model.predict(X_val)

print("\nBest model on validation set:")
print(f"  R²:   {r2_score(y_val, y_pred_val):.4f}")
print(f"  MAE:  {mean_absolute_error(y_val, y_pred_val):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_val, y_pred_val)):.4f}")


Full model comparison (test set):
                      Model     R²     MAE    RMSE
 XGBoost (RandomizedSearch) 0.5776 50.2285 71.0623
                   CatBoost 0.5739 50.5769 71.3711
LightGBM (RandomizedSearch) 0.5731 50.6355 71.4446
CatBoost (RandomizedSearch) 0.5698 50.9355 71.7137
                    XGBoost 0.5695 50.8892 71.7413
       XGBoost (GridSearch) 0.5681 51.0381 71.8571
      CatBoost (GridSearch) 0.5674 51.1424 71.9131
                   LightGBM 0.5600 51.7534 72.5328
      LightGBM (GridSearch) 0.5588 51.8351 72.6241
              Random Forest 0.5526 52.4020 73.1345
          Linear Regression 0.5069 56.3054 76.7839
                        KNN 0.3098 69.3095 90.8418

Best model on validation set:
  R²:   0.5767
  MAE:  50.6216
  RMSE: 71.3291


In [16]:
# ── CHANGED ───────────────────────────────────────────────────
# CHANGE: save full pipeline as best model (includes encoder + scaler)
df_final.to_csv(data_path / "model_comparison_results.csv", index=False)
joblib.dump(best_model, data_path / "best_model.pkl")  # CHANGED: full pipeline saved
print("Results and best model pipeline saved!")

Results and best model pipeline saved!


In [ ]:
#OLD COMMENTS FROM THE LAST NOTEBOOK, NEED TO CORRECT (MAYBE)

Entrenando XGBoost con GridSearch:


ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/xgboost/data.py", line 408, in pandas_feature_info
    new_feature_types.append(_pandas_dtype_mapper[dtype.name])
KeyError: 'object'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/kilian/venvs/mle_liora_london_firefighter/lib/python3.10/site-packages/xgboost/core.py", line 751, in inne

In [32]:
#we continue in the next notebook, so we save the results and the best model using joblib

# Save all results to CSV
df_final.to_csv("/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/model_comparison_results.csv", index=False)


#Save the best model

import joblib

joblib.dump(random_xgb, "/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/best_model.pkl")


['/mnt/e/Users/KilianAT/Documents/Estudios/Weiterbildung/Data Science/Proyecto/mle_liora_london_firefighter/data/best_model.pkl']